# Statistics and Data Science Foundations
## Descriptive Stats · Probability · Hypothesis Testing · EDA · Feature Engineering

---

## Table of Contents

1. Descriptive Statistics
2. Probability Distributions
3. Hypothesis Testing
4. Correlation and Relationships
5. Exploratory Data Analysis (EDA) Pipeline
6. Feature Engineering
7. Feature Selection
8. Dimensionality Reduction

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)
rng = np.random.default_rng(42)

# --- Shared Dataset ---
n = 500
df = pd.DataFrame({
    'age':        rng.integers(22, 65, n),
    'tenure':     rng.integers(1, 20, n),
    'salary':     rng.integers(35000, 150000, n).astype(float),
    'score':      rng.normal(72, 15, n).clip(0, 100),
    'department': rng.choice(['Engineering','Marketing','Sales','HR','Finance'], n),
    'gender':     rng.choice(['M','F'], n),
    'promoted':   rng.choice([0, 1], n, p=[0.7, 0.3])
})
# Inject realistic correlation: salary grows with tenure
df['salary'] = df['salary'] + df['tenure'] * 2000 + df['age'] * 500
df['salary'] = df['salary'].clip(35000, 200000)

# Add some missing values and outliers for EDA
df.loc[rng.choice(n, 25, replace=False), 'score']   = np.nan
df.loc[rng.choice(n, 15, replace=False), 'tenure']  = np.nan
df.loc[rng.integers(0, 5), 'salary'] = rng.integers(400000, 600000, 5)  # outliers

print('Dataset shape:', df.shape)
print(df.dtypes)
print(df.head())

# Section 1 — Descriptive Statistics

## Concept

Descriptive statistics summarize the main features of a dataset without making inferences about a larger population.
They answer: **What does the data look like?**

## Technical Deep Dive

**Central tendency:** Mean, Median, Mode

**Spread:** Variance, Standard deviation, IQR, Range

**Shape:** Skewness, Kurtosis

| Measure | Formula | Sensitive to outliers? |
|---|---|---|
| Mean | `sum(x) / n` | Yes |
| Median | Middle value | No |
| Variance | `E[(x - μ)²]` | Yes |
| Std dev | `sqrt(variance)` | Yes |
| IQR | `Q3 - Q1` | No |

**Skewness:**
- Positive skew: tail on the right (mean > median)
- Negative skew: tail on the left (mean < median)

**Kurtosis:**
- High kurtosis (leptokurtic): heavy tails, more outliers
- Low kurtosis (platykurtic): thin tails


In [ ]:
# Comprehensive descriptive stats
print('=== pandas describe ===')
print(df.describe().round(2))

In [ ]:
salary = df['salary'].dropna()

def describe_variable(series, name):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    print(f'--- {name} ---')
    print(f'  Count:    {len(series)}')
    print(f'  Mean:     {series.mean():.2f}')
    print(f'  Median:   {series.median():.2f}')
    print(f'  Std:      {series.std():.2f}')
    print(f'  Variance: {series.var():.2f}')
    print(f'  IQR:      {q3 - q1:.2f}')
    print(f'  Min/Max:  {series.min():.2f} / {series.max():.2f}')
    print(f'  Skewness: {series.skew():.4f}')
    print(f'  Kurtosis: {series.kurtosis():.4f}')
    print(f'  CV:       {series.std()/series.mean()*100:.1f}%  (coefficient of variation)')

describe_variable(salary, 'Salary')
describe_variable(df['score'].dropna(), 'Score')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histogram with mean/median lines
salary_clean = salary[salary < 300000]  # exclude outliers for viz
axes[0].hist(salary_clean, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(salary_clean.mean(),   color='red',    linestyle='--', label=f'Mean: ${salary_clean.mean()/1000:.0f}k')
axes[0].axvline(salary_clean.median(), color='orange', linestyle='--', label=f'Median: ${salary_clean.median()/1000:.0f}k')
axes[0].set_title('Salary Distribution')
axes[0].set_xlabel('Salary ($)')
axes[0].legend(fontsize=9)

# Box plot
axes[1].boxplot(salary_clean, patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Salary Box Plot')
axes[1].set_ylabel('Salary ($)')
axes[1].set_xticks([])

# Q-Q plot to test normality
score_clean = df['score'].dropna()
stats.probplot(score_clean, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot: Score vs Normal')

plt.tight_layout(); plt.show()

## Exercises

1. Compute the 10th, 25th, 50th, 75th, and 90th percentiles of `salary`.
2. Compute the coefficient of variation (CV = std/mean) for all numeric columns.
3. Detect outliers using the IQR method: flag values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`.
4. Compare the mean vs median of `salary` — what does the difference tell you about the distribution?

## Mini Challenge

Build a `full_describe(df)` function that returns a DataFrame with: count, missing%, mean, median, std, min, max, IQR, skewness, kurtosis, and outlier count (IQR method) for all numeric columns.

## Best Practices

- Always report median alongside mean for skewed distributions.
- Use IQR or percentile-based methods (not std) to detect outliers in skewed data.
- CV (std/mean) allows comparing spread across variables with different scales.

## Summary

- Central tendency: mean (outlier-sensitive) vs median (robust).
- Spread: std, IQR — know which to use and when.
- Shape: skewness (asymmetry), kurtosis (tail heaviness).
- Q-Q plot: visual normality test.

---


### Exercise and Challenge Solutions — Section 1


In [ ]:
# Exercise 1: percentiles
# YOUR CODE HERE
# Exercise 2: CV
# YOUR CODE HERE
# Exercise 3: IQR outliers
def iqr_outliers(series):
    # YOUR CODE HERE
    pass

In [ ]:
def full_describe(df):
    num = df.select_dtypes(include=np.number)
    result = []
    for col in num.columns:
        s = num[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        outliers = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
        result.append({
            'column':    col,
            'count':     len(s),
            'missing_%': round(df[col].isna().mean() * 100, 1),
            'mean':      round(s.mean(), 2),
            'median':    round(s.median(), 2),
            'std':       round(s.std(), 2),
            'min':       round(s.min(), 2),
            'max':       round(s.max(), 2),
            'IQR':       round(iqr, 2),
            'skewness':  round(s.skew(), 3),
            'kurtosis':  round(s.kurtosis(), 3),
            'outliers':  int(outliers)
        })
    return pd.DataFrame(result).set_index('column')

print(full_describe(df))

# Section 2 — Probability Distributions

## Concept

A probability distribution describes how values of a random variable are spread.
Knowing which distribution fits your data guides modeling choices.

## Technical Deep Dive

| Distribution | Use Case | Parameters |
|---|---|---|
| Normal | Measurement errors, heights | μ, σ |
| Binomial | Count of successes in n trials | n, p |
| Poisson | Event counts per time interval | λ |
| Exponential | Time between events | λ (rate) |
| Uniform | Equal probability in [a, b] | a, b |
| Log-normal | Right-skewed (income, prices) | μ, σ of log(x) |
| Beta | Probabilities in [0, 1] | α, β |
| Chi-squared | Sum of squared normals, goodness of fit | k (df) |

**scipy.stats** provides `pdf`, `cdf`, `ppf` (inverse CDF), `rvs` (random samples) for all.


In [ ]:
from scipy import stats as st

x = np.linspace(-4, 4, 400)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Normal
for mu, sigma, c in [(0,1,'steelblue'), (0,2,'tomato'), (1,1,'green')]:
    axes[0,0].plot(x, st.norm.pdf(x, mu, sigma), label=f'μ={mu},σ={sigma}', color=c)
axes[0,0].set_title('Normal Distribution PDF'); axes[0,0].legend(fontsize=8)

# Binomial
k = np.arange(0, 21)
for n_t, p, c in [(20,0.3,'steelblue'), (20,0.5,'tomato'), (20,0.7,'green')]:
    axes[0,1].bar(k - 0.25, st.binom.pmf(k, n_t, p), width=0.25, alpha=0.7, label=f'p={p}', color=c)
axes[0,1].set_title('Binomial PMF (n=20)'); axes[0,1].legend(fontsize=8)

# Poisson
k_p = np.arange(0, 16)
for lam, c in [(2,'steelblue'), (5,'tomato'), (10,'green')]:
    axes[0,2].plot(k_p, st.poisson.pmf(k_p, lam), 'o-', label=f'λ={lam}', color=c)
axes[0,2].set_title('Poisson PMF'); axes[0,2].legend(fontsize=8)

# Exponential
x_e = np.linspace(0, 5, 300)
for rate, c in [(0.5,'steelblue'), (1,'tomato'), (2,'green')]:
    axes[1,0].plot(x_e, st.expon.pdf(x_e, scale=1/rate), label=f'rate={rate}', color=c)
axes[1,0].set_title('Exponential PDF'); axes[1,0].legend(fontsize=8)

# Log-normal
x_ln = np.linspace(0.01, 10, 400)
for s, c in [(0.5,'steelblue'), (1,'tomato'), (1.5,'green')]:
    axes[1,1].plot(x_ln, st.lognorm.pdf(x_ln, s), label=f's={s}', color=c)
axes[1,1].set_title('Log-normal PDF'); axes[1,1].legend(fontsize=8)

# Beta
x_b = np.linspace(0.01, 0.99, 300)
for a, b, c in [(2,5,'steelblue'), (5,2,'tomato'), (2,2,'green')]:
    axes[1,2].plot(x_b, st.beta.pdf(x_b, a, b), label=f'α={a},β={b}', color=c)
axes[1,2].set_title('Beta PDF'); axes[1,2].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# Practical scipy.stats operations
norm_dist = st.norm(loc=70000, scale=20000)

print('P(salary < 80000):', norm_dist.cdf(80000).round(4))
print('P(salary > 100000):', (1 - norm_dist.cdf(100000)).round(4))
print('P(60k < salary < 90k):', (norm_dist.cdf(90000) - norm_dist.cdf(60000)).round(4))
print('90th percentile salary:', norm_dist.ppf(0.90).round(0))

# Fit a distribution to data
score_clean = df['score'].dropna()
mu_fit, sigma_fit = st.norm.fit(score_clean)
print(f'\nFitted Normal to Score: μ={mu_fit:.2f}, σ={sigma_fit:.2f}')

# Normality test
stat, p = st.shapiro(score_clean[:50])  # Shapiro-Wilk (works best < 5000 samples)
print(f'Shapiro-Wilk test: stat={stat:.4f}, p={p:.4f}')
print('Conclusion:', 'Normal' if p > 0.05 else 'Not normal', f'(α=0.05)')

## Exercises

1. A website has 1000 visitors and 5% conversion rate. What is the probability of exactly 60 conversions? More than 70? (Binomial)
2. A call center receives 10 calls per hour on average. What is P(more than 15 calls in one hour)? (Poisson)
3. Fit a log-normal distribution to `salary` and compare its PDF to the actual histogram.
4. Use `scipy.stats.kstest` to test if `score` follows a normal distribution.

## Mini Challenge

Monte Carlo simulation: simulate 10,000 samples of total monthly revenue where
daily revenue ~ Normal(10000, 2000) and there are Poisson(28) business days per month.
Report the 5th and 95th percentile of monthly revenue.

## Summary

- `scipy.stats.dist` provides `pdf`, `cdf`, `ppf`, `rvs`, and `fit`.
- Normal: symmetric, bell-shaped — most common assumption in ML.
- Poisson: event counts; Exponential: inter-event times.
- Log-normal: right-skewed data like salaries, prices.

---


### Exercise and Challenge Solutions — Section 2


In [ ]:
# Exercise 1: Binomial
# YOUR CODE HERE
# Exercise 2: Poisson
# YOUR CODE HERE
# Exercise 4: KS test
# YOUR CODE HERE

In [ ]:
# Mini Challenge: Monte Carlo revenue simulation
# YOUR CODE HERE

# Section 3 — Hypothesis Testing

## Concept

Hypothesis testing provides a framework for making decisions from data under uncertainty.

**Process:**
1. State H₀ (null) and H₁ (alternative) hypotheses.
2. Choose significance level α (commonly 0.05).
3. Select the appropriate test.
4. Compute test statistic and p-value.
5. Reject H₀ if p < α.

## Technical Deep Dive

| Test | Use Case | Function |
|---|---|---|
| One-sample t-test | Mean vs known value | `stats.ttest_1samp` |
| Two-sample t-test | Compare two means | `stats.ttest_ind` |
| Paired t-test | Before/after same subjects | `stats.ttest_rel` |
| Welch t-test | Unequal variances | `ttest_ind(equal_var=False)` |
| Mann-Whitney U | Non-parametric two-sample | `stats.mannwhitneyu` |
| Chi-squared | Categorical independence | `stats.chi2_contingency` |
| ANOVA | Compare 3+ means | `stats.f_oneway` |
| Kruskal-Wallis | Non-parametric ANOVA | `stats.kruskal` |
| Shapiro-Wilk | Normality | `stats.shapiro` |
| Levene | Equality of variances | `stats.levene` |

**p-value:** probability of observing data as extreme as the sample, assuming H₀ is true.

**Effect size matters** — a statistically significant result may not be practically significant.


In [ ]:
from scipy import stats as st

# One-sample t-test: is mean salary different from 80000?
salary_clean = df['salary'][df['salary'] < 300000]
t_stat, p_val = st.ttest_1samp(salary_clean, popmean=80000)
print(f'One-sample t-test (H0: mean=80000):')
print(f'  t={t_stat:.4f}, p={p_val:.4f}')
print(f'  Result: {"Reject H0" if p_val < 0.05 else "Fail to reject H0"} (α=0.05)')

# Two-sample t-test: do M and F have different salaries?
sal_m = df[df['gender']=='M']['salary'].dropna()
sal_m = sal_m[sal_m < 300000]
sal_f = df[df['gender']=='F']['salary'].dropna()
sal_f = sal_f[sal_f < 300000]

# Levene test for equal variances first
lev_stat, lev_p = st.levene(sal_m, sal_f)
print(f'\nLevene test (equal variances): stat={lev_stat:.4f}, p={lev_p:.4f}')

# Use Welch t-test (safe regardless of variance equality)
t2, p2 = st.ttest_ind(sal_m, sal_f, equal_var=False)
print(f'\nWelch t-test (H0: μ_M = μ_F):')
print(f'  M mean: ${sal_m.mean():,.0f}   F mean: ${sal_f.mean():,.0f}')
print(f'  t={t2:.4f}, p={p2:.4f}')
print(f'  Result: {"Significant difference" if p2 < 0.05 else "No significant difference"}')

In [ ]:
# Chi-squared test: is promotion independent of gender?
contingency = pd.crosstab(df['gender'], df['promoted'])
print('Contingency table (gender x promoted):')
print(contingency)

chi2, p_chi2, dof, expected = st.chi2_contingency(contingency)
print(f'\nChi2={chi2:.4f}, p={p_chi2:.4f}, dof={dof}')
print(f'Result: {"Dependent (reject H0)" if p_chi2 < 0.05 else "Independent (fail to reject H0)"}')

# One-way ANOVA: salary across departments
dept_salaries = [
    df[(df['department']==d) & (df['salary'] < 300000)]['salary'].dropna()
    for d in df['department'].unique()
]
f_stat, p_anova = st.f_oneway(*dept_salaries)
print(f'\nANOVA (salary ~ department):')
print(f'  F={f_stat:.4f}, p={p_anova:.4f}')
print(f'  Result: {"At least one dept differs" if p_anova < 0.05 else "No difference"}')

In [ ]:
# Effect size — Cohen's d
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

d = cohens_d(sal_m, sal_f)
print(f"Cohen's d (M vs F salary): {d:.4f}")
magnitude = 'small' if abs(d) < 0.2 else ('medium' if abs(d) < 0.5 else 'large')
print(f'Effect size magnitude: {magnitude}')

# Multiple testing correction — Bonferroni
depts = df['department'].unique()
pairs = [(d1, d2) for i, d1 in enumerate(depts) for d2 in depts[i+1:]]
p_values = []
for d1, d2 in pairs:
    s1 = df[(df['department']==d1) & (df['salary']<300000)]['salary'].dropna()
    s2 = df[(df['department']==d2) & (df['salary']<300000)]['salary'].dropna()
    _, p = st.ttest_ind(s1, s2, equal_var=False)
    p_values.append(p)

alpha = 0.05
bonferroni_alpha = alpha / len(pairs)
print(f'\nPairwise t-tests ({len(pairs)} comparisons):')
print(f'Bonferroni-corrected α: {bonferroni_alpha:.4f}')
for (d1, d2), p in zip(pairs, p_values):
    sig = '*' if p < bonferroni_alpha else ''
    print(f'  {d1} vs {d2}: p={p:.4f} {sig}')

## Exercises

1. Test whether the mean performance `score` is significantly above 70 (one-sample t-test).
2. Test if promoted employees have significantly higher `score` than non-promoted (two-sample).
3. Run a Kruskal-Wallis test on `score` across departments (non-parametric ANOVA alternative).
4. Compute and interpret Cohen's d for the promoted vs non-promoted score comparison.

## Mini Challenge

Implement an A/B test simulator: generate two groups (control/treatment) with given true conversion rates, run a chi-squared test, and report: p-value, effect size (relative lift), and whether you'd reject H₀. Run it 1000 times with `p_control=0.05, p_treatment=0.065, n=5000` per group and report the empirical power.

## Summary

- Choose the test based on: data type, number of groups, independence assumption, normality.
- p-value < α means reject H₀, not that H₁ is true.
- Always report effect size (Cohen's d, odds ratio) alongside p-values.
- Multiple testing requires correction (Bonferroni, Benjamini-Hochberg).

---


### Exercise and Challenge Solutions — Section 3


In [ ]:
# Exercise 1
# YOUR CODE HERE
# Exercise 2
# YOUR CODE HERE
# Exercise 3: Kruskal-Wallis
# YOUR CODE HERE

In [ ]:
# Mini Challenge: A/B test power simulation
# YOUR CODE HERE

# Section 4 — Correlation and Relationships

## Concept

Correlation measures the strength and direction of the linear (or monotonic) relationship between two variables.

## Technical Deep Dive

| Method | Range | Assumes | Use When |
|---|---|---|---|
| Pearson | [-1, 1] | Linear, normal | Continuous, linear |
| Spearman | [-1, 1] | Monotonic | Ordinal or non-linear |
| Kendall τ | [-1, 1] | Monotonic | Small samples, many ties |
| Point-biserial | [-1, 1] | Linear | Continuous + binary |

**Correlation ≠ Causation.** Always check for confounding variables.

**Anscombe's Quartet:** four datasets with the same summary stats but completely different distributions.


In [ ]:
df_clean = df[df['salary'] < 300000].dropna()

# Pearson, Spearman, Kendall
for method in ['pearson', 'spearman', 'kendall']:
    r = df_clean[['salary','tenure','score','age']].corr(method=method)
    print(f'\n=== {method.capitalize()} Correlation ===')
    print(r.round(3))

In [ ]:
# Heatmap with significance stars
num_cols_clean = df_clean[['salary','tenure','score','age']]
corr_matrix  = num_cols_clean.corr()

# Compute p-values
n_c = len(num_cols_clean)
p_matrix = pd.DataFrame(np.ones((4,4)), columns=num_cols_clean.columns, index=num_cols_clean.columns)
for c1 in num_cols_clean.columns:
    for c2 in num_cols_clean.columns:
        if c1 != c2:
            _, p = st.pearsonr(num_cols_clean[c1], num_cols_clean[c2])
            p_matrix.loc[c1, c2] = p

# Annotations: correlation + significance
annot = corr_matrix.round(2).astype(str)
for c1 in annot.index:
    for c2 in annot.columns:
        if c1 != c2:
            stars = '***' if p_matrix.loc[c1,c2] < 0.001 else ('**' if p_matrix.loc[c1,c2] < 0.01 else ('*' if p_matrix.loc[c1,c2] < 0.05 else ''))
            annot.loc[c1, c2] = str(corr_matrix.loc[c1,c2].round(2)) + stars

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=annot, fmt='', cmap='coolwarm', center=0,
            square=True, linewidths=1, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Matrix (* p<0.05, ** p<0.01, *** p<0.001)')
plt.tight_layout(); plt.show()

## Summary

- Pearson for linear relationships; Spearman for monotonic/non-normal data.
- Always visualize the scatter before interpreting a correlation coefficient.
- Correlation is symmetric and not causal.
- High correlation between features = multicollinearity (problem for linear models).

---


# Section 5 — Exploratory Data Analysis (EDA) Pipeline

## Concept

EDA is the systematic exploration of a dataset before modeling:
understand the structure, find patterns, spot anomalies.

## EDA Pipeline

1. **Shape and types** — `.shape`, `.dtypes`, `.info()`
2. **Missing data** — `.isna().sum()`, heatmap
3. **Univariate** — histograms, box plots per variable
4. **Bivariate** — scatter matrix, correlation, group comparisons
5. **Outlier detection** — IQR, Z-score, visual inspection
6. **Target analysis** — class balance, target correlations
7. **Hypotheses** — note findings to guide modeling


In [ ]:
# Step 1-2: Overview
print('=== Shape ===')
print(df.shape)
print('\n=== Dtypes ===')
print(df.dtypes)
print('\n=== Missing Data ===')
missing = df.isna().sum()
missing_pct = df.isna().mean() * 100
print(pd.concat([missing, missing_pct.round(1)], axis=1, keys=['count','%']))

In [ ]:
# Missing data heatmap
fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(df.isna().T, cmap='YlOrRd', cbar=False, ax=ax, yticklabels=True)
ax.set_title('Missing Data Pattern (yellow = missing)')
ax.set_xlabel('Row index')
plt.tight_layout(); plt.show()

In [ ]:
# Univariate distributions of all numeric columns
num_cols = df.select_dtypes(include=np.number).columns.tolist()
n_cols = len(num_cols)
fig, axes = plt.subplots(2, (n_cols+1)//2, figsize=(14, 6))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    data = data[data < data.quantile(0.99)]  # clip extreme outliers for viz
    axes[i].hist(data, bins=25, color='steelblue', alpha=0.8, edgecolor='white')
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5)
    axes[i].set_title(f'{col} (skew={df[col].skew():.2f})')
    axes[i].set_xlabel(col)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(); plt.show()

In [ ]:
# Bivariate: salary vs numeric features by promotion status
df_plot = df[df['salary'] < 300000].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

features = ['age', 'tenure', 'score']
for ax, feat in zip(axes, features):
    for promoted, color, label in [(0,'steelblue','Not Promoted'), (1,'tomato','Promoted')]:
        subset = df_plot[df_plot['promoted']==promoted]
        ax.scatter(subset[feat], subset['salary'], alpha=0.4, s=20, color=color, label=label)
    ax.set_xlabel(feat); ax.set_ylabel('Salary')
    ax.set_title(f'Salary vs {feat}')
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# Target analysis: class balance + group comparisons
print('=== Target (promoted) class balance ===')
balance = df['promoted'].value_counts(normalize=True)
print(balance.round(3))
print('Imbalance ratio:', f'{balance[0]/balance[1]:.2f}:1')

# Group stats by target
print('\n=== Feature means by promoted status ===')
df_clean2 = df[df['salary'] < 300000]
print(df_clean2.groupby('promoted')[['age','tenure','salary','score']].mean().round(1))

## Summary

- EDA is hypothesis generation, not hypothesis testing.
- Always visualize; numbers alone hide shape and outliers.
- Check class balance before classification — imbalanced targets need special handling.
- Document EDA findings as hypotheses to test during modeling.

---


# Section 6 — Feature Engineering

## Concept

Feature engineering transforms raw variables into representations that help ML models learn better.
It is often the highest-leverage activity in a data science project.

## Technical Deep Dive

| Technique | Description |
|---|---|
| Scaling | Normalize numeric ranges |
| Encoding | Convert categories to numbers |
| Binning | Discretize continuous variables |
| Log transform | Reduce right skew |
| Polynomial features | Capture non-linear relationships |
| Interaction features | Product of two features |
| Date/Time features | Extract year, month, day, weekday |
| Aggregation features | Group-level stats (mean per group) |
| Target encoding | Replace category with mean target |


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

df_feat = df[df['salary'] < 300000].dropna().copy().reset_index(drop=True)

# Scaling comparison
salary_raw = df_feat['salary'].values.reshape(-1, 1)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler()   # uses median/IQR — robust to outliers
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
axes[0].hist(salary_raw, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Raw Salary')

for ax, (name, scaler) in zip(axes[1:], scalers.items()):
    scaled = scaler.fit_transform(salary_raw)
    ax.hist(scaled, bins=30, color='teal', alpha=0.8, edgecolor='white')
    ax.set_title(name)

plt.tight_layout(); plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# 1. One-hot encoding (get_dummies)
df_ohe = pd.get_dummies(df_feat, columns=['department','gender'], drop_first=True)
print('One-hot encoded shape:', df_ohe.shape)
print(df_ohe.filter(like='department').head())

# 2. Label encoding (ordinal)
le = LabelEncoder()
df_feat['dept_label'] = le.fit_transform(df_feat['department'])
print('\nLabel encoded dept:', dict(zip(le.classes_, le.transform(le.classes_))))

# 3. Target encoding (mean target per category)
target_enc = df_feat.groupby('department')['promoted'].mean()
df_feat['dept_target_enc'] = df_feat['department'].map(target_enc)
print('\nTarget encoding (dept → mean promotion rate):')
print(target_enc.round(3))

In [ ]:
# Log transform for skewed variables
df_feat['log_salary'] = np.log1p(df_feat['salary'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_feat['salary'], bins=30, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title(f'Raw Salary (skew={df_feat["salary"].skew():.2f})')
axes[1].hist(df_feat['log_salary'], bins=30, color='teal', alpha=0.8, edgecolor='white')
axes[1].set_title(f'Log(1+Salary) (skew={df_feat["log_salary"].skew():.2f})')
plt.tight_layout(); plt.show()

# Interaction and polynomial features
df_feat['salary_per_year'] = df_feat['salary'] / (df_feat['tenure'] + 1)
df_feat['tenure_squared']  = df_feat['tenure'] ** 2
df_feat['age_tenure_inter']= df_feat['age'] * df_feat['tenure']

# Binning tenure into experience brackets
df_feat['experience'] = pd.cut(
    df_feat['tenure'], bins=[0, 3, 7, 15, 25],
    labels=['Junior', 'Mid', 'Senior', 'Principal']
)

print('New features created:')
print(df_feat[['salary','log_salary','salary_per_year','tenure_squared','experience']].head())

In [ ]:
# Date/Time feature extraction
orders = pd.DataFrame({
    'order_date': pd.date_range('2023-01-01', periods=100, freq='D'),
    'amount': rng.integers(50, 500, 100)
})

orders['year']       = orders['order_date'].dt.year
orders['month']      = orders['order_date'].dt.month
orders['dayofweek']  = orders['order_date'].dt.dayofweek  # 0=Monday
orders['is_weekend'] = orders['dayofweek'].isin([5, 6]).astype(int)
orders['quarter']    = orders['order_date'].dt.quarter
orders['week_no']    = orders['order_date'].dt.isocalendar().week

print(orders.head(8))

## Exercises

1. Apply Box-Cox transformation to `salary` using `scipy.stats.boxcox` and compare skewness.
2. Create a `high_performer` feature: 1 if score > 85th percentile, else 0.
3. Implement leave-one-out target encoding for the `department` column.
4. Create cyclical encoding for `month` (sin/cos encoding to preserve circular nature).

## Mini Challenge

Build a `FeatureEngineer` class that applies: log transform, StandardScaler, one-hot encoding, and interaction features — all in a `fit_transform(df)` method that returns a ready-to-use feature matrix.

## Summary

- Scale numeric features — tree models don't need it, linear/distance-based models do.
- Use RobustScaler when outliers are present.
- One-hot for low-cardinality categoricals; target encoding for high-cardinality.
- Log/Box-Cox transforms reduce right skew — important for linear models.

---


### Exercise and Challenge Solutions — Section 6


In [ ]:
# Exercise 1: Box-Cox
# YOUR CODE HERE
# Exercise 2
# YOUR CODE HERE
# Exercise 4: cyclical month encoding
# YOUR CODE HERE

# Section 7 — Feature Selection

## Concept

Feature selection removes irrelevant or redundant features to:
- Reduce overfitting
- Improve model speed
- Increase interpretability

## Technical Deep Dive

| Method | Type | Technique |
|---|---|---|
| Variance threshold | Filter | Remove near-zero variance |
| Correlation filter | Filter | Drop highly correlated pairs |
| Chi-squared | Filter | Categorical target |
| Mutual information | Filter | Captures non-linear dependence |
| RFE | Wrapper | Recursive feature elimination |
| L1 regularization | Embedded | Lasso zeroes out weak features |
| Feature importance | Embedded | Tree-based importance |
| Permutation importance | Post-hoc | Model-agnostic |


In [ ]:
from sklearn.feature_selection import (VarianceThreshold, SelectKBest,
                                        mutual_info_classif, RFE)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Prepare feature matrix
df_sel = df_feat.copy()
df_sel = pd.get_dummies(df_sel, columns=['department','gender'], drop_first=True)
df_sel = df_sel.select_dtypes(include=np.number).dropna()

# Exclude target-derived features
exclude = ['promoted','dept_target_enc']
feat_cols = [c for c in df_sel.columns if c not in exclude + ['experience']]

X = df_sel[feat_cols]
y = df_sel['promoted']

print('Feature matrix shape:', X.shape)
print('Features:', feat_cols)

In [ ]:
# 1. Variance threshold
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)
removed_vt = [c for c, keep in zip(X.columns, vt.get_support()) if not keep]
print(f'Variance threshold: removed {len(removed_vt)} features: {removed_vt}')

# 2. Correlation filter
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.90)]
print(f'High correlation (>0.90): drop {to_drop_corr}')

# 3. Mutual information
X_scaled = StandardScaler().fit_transform(X)
mi_scores = mutual_info_classif(X_scaled, y, random_state=42)
mi_df = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print('\nMutual Information scores:')
print(mi_df.round(4))

In [ ]:
# 4. Random Forest feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(importance_df['feature'], importance_df['importance'], color='steelblue', alpha=0.85)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importance')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

# Select top N features
top_k = 6
top_features = importance_df.head(top_k)['feature'].tolist()
print(f'\nTop {top_k} features:', top_features)

## Summary

- Start with filter methods (fast, model-agnostic).
- Use embedded methods (L1, tree importance) for efficient selection within model training.
- Wrapper methods (RFE) are expensive but optimal for the chosen model.
- Always evaluate selection on validation data, not training data.

---


# Section 8 — Dimensionality Reduction

## Concept

Dimensionality reduction compresses many features into fewer dimensions while preserving
maximum information. Used for visualization, noise reduction, and speeding up models.

## Technical Deep Dive

| Method | Type | Best For |
|---|---|---|
| PCA | Linear, unsupervised | Variance preservation, preprocessing |
| t-SNE | Non-linear, unsupervised | 2D/3D visualization of clusters |
| UMAP | Non-linear, unsupervised | Fast, topology-preserving viz |
| LDA | Linear, supervised | Classification preprocessing |
| SVD/TruncatedSVD | Linear | Sparse data (text, recommendations) |
| Autoencoders | Non-linear, deep learning | Complex high-dimensional data |


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# PCA on employee features
X_std = StandardScaler().fit_transform(X)

pca = PCA(n_components=min(X.shape[1], 8))
X_pca = pca.fit_transform(X_std)

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

evr = pca.explained_variance_ratio_
axes[0].bar(range(1, len(evr)+1), evr, color='steelblue', alpha=0.8)
axes[0].plot(range(1, len(evr)+1), np.cumsum(evr), 'o-', color='tomato', label='Cumulative')
axes[0].axhline(0.90, color='gray', linestyle='--', label='90% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Scree Plot')
axes[0].legend()

# 2D PCA scatter colored by promoted
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_std)
scatter = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='coolwarm', alpha=0.6, s=25)
axes[1].set_title(f'PCA 2D (explains {pca2.explained_variance_ratio_.sum()*100:.1f}% variance)')
axes[1].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter, ax=axes[1], label='Promoted')

plt.tight_layout(); plt.show()

In [ ]:
# PCA Loadings — which features contribute to each PC?
loadings = pd.DataFrame(
    pca2.components_.T,
    index=X.columns,
    columns=['PC1', 'PC2']
).sort_values('PC1', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pc in zip(axes, ['PC1','PC2']):
    loadings[pc].sort_values().plot.barh(ax=ax, color='steelblue', alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{pc} Loadings')
    ax.set_xlabel('Loading')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.manifold import TSNE

# t-SNE — non-linear 2D embedding
# Use subset for speed
sample_idx = rng.choice(len(X_std), size=min(300, len(X_std)), replace=False)
X_sample = X_std[sample_idx]
y_sample = y.values[sample_idx]
dept_sample = df_feat['department'].values[:len(X_std)][sample_idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_sample)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colored by promoted
scatter = axes[0].scatter(X_tsne[:,0], X_tsne[:,1], c=y_sample, cmap='coolwarm', alpha=0.7, s=30)
axes[0].set_title('t-SNE colored by Promoted')
axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=axes[0])

# Colored by department
dept_labels = pd.Categorical(dept_sample)
scatter2 = axes[1].scatter(X_tsne[:,0], X_tsne[:,1],
                            c=dept_labels.codes, cmap='Set1', alpha=0.7, s=30)
axes[1].set_title('t-SNE colored by Department')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')

# Legend for departments
for code, dept in enumerate(dept_labels.categories):
    axes[1].scatter([], [], c=[plt.cm.Set1(code/8)], label=dept, s=40)
axes[1].legend(fontsize=8, loc='upper right')

plt.tight_layout(); plt.show()

## Exercises

1. Determine how many PCA components are needed to explain 95% of variance.
2. Reconstruct the original data from 3 PCA components and compute the reconstruction error.
3. Run t-SNE with different `perplexity` values (5, 30, 50) and compare the embeddings.
4. Apply `TruncatedSVD` from sklearn to a sparse matrix of random data.

## Mini Challenge

Build a complete preprocessing + PCA pipeline using `sklearn.pipeline.Pipeline`:
impute → scale → PCA(95% variance) → return transformed matrix with correct shape.
Fit on 80% of data, transform the remaining 20% and report the explained variance.

## Summary

- PCA: linear, global structure, fast — best for preprocessing.
- t-SNE: non-linear, local structure, slow — best for visualization only.
- Always standardize before PCA/t-SNE.
- PCA components are orthogonal — no multicollinearity problem.

---


### Exercise and Challenge Solutions — Section 8


In [ ]:
# Exercise 1: components for 95% variance
# YOUR CODE HERE
# Exercise 2: reconstruction error
# YOUR CODE HERE

In [ ]:
# Mini Challenge: sklearn Pipeline
# YOUR CODE HERE

# Course Summary

| Section | Key Skills |
|---|---|
| 1. Descriptive Stats | Mean/Median/Std/IQR/Skewness/Kurtosis, Q-Q plot |
| 2. Distributions | scipy.stats: pdf/cdf/ppf/rvs/fit, Monte Carlo |
| 3. Hypothesis Testing | t-test, chi-squared, ANOVA, effect size, power |
| 4. Correlation | Pearson/Spearman, multicollinearity, p-values |
| 5. EDA Pipeline | Shape → Missing → Univariate → Bivariate → Target |
| 6. Feature Engineering | Scaling, encoding, log, interactions, datetime |
| 7. Feature Selection | Filter, embedded (importance), wrapper (RFE) |
| 8. Dimensionality Reduction | PCA, t-SNE, sklearn Pipeline |

## Next Steps

- **`machine_learning_course.ipynb`** — Scikit-learn: supervised, unsupervised, evaluation
- **`deep_learning_course.ipynb`** — Neural Networks with PyTorch

---
